# Mixed-source causal fine-tuning experiments

This notebook uses [Yelp Polarity](https://huggingface.co/datasets/fancyzhx/yelp_polarity) and [Amazon Polarity](https://huggingface.co/datasets/fancyzhx/amazon_polarity) to construct a source-correlated sentiment task.

For each source and class, `N = 5000` examples are sampled with `data_seed = 5` and split **80/20** into training and validation. A separate pool of 2,000 examples per class and source is sampled from the test split with seed 42. After source mixing, the defaults produce 8,000 training examples, 2,000 validation examples, and 4,000 examples per test condition.

Run the setup cells in order before a model section. Results are written under `./results/sentiment/`, relative to the notebook's working directory.


In [ ]:
from transformers import set_seed
from datasets import load_dataset, concatenate_datasets, DatasetDict, ClassLabel
import os
import torch.nn as nn
import evaluate
import numpy as np
import json
import copy
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torch
import torch.optim as optim
import random
import re
from torch.utils.data import Dataset, DataLoader
import torch
from transformers import DataCollatorWithPadding
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer

In [ ]:
def realworld_data_sampler(dataset, n_samples, sampler_seed, test_per_class=2000, max_search_words=30, string='amazon'):
    
    # Gather the number of labels from the test dataset with predefined train, validation ratio
    num_cls = len(set(dataset['test']['label']))
    ratio = 0.8

    print(f'Generate synthetic data of {num_cls} classes with random seed {sampler_seed}, each class of {int(ratio * n_samples)} training, {n_samples - int(ratio * n_samples)} validation and {test_per_class} test data.')

    train_datasets = []
    val_datasets = []
    test_datasets = []

    def filter_sentence(example):
        words = ' '.join(example['text'].split())
        search_words = words[:max_search_words] 
        count_the = len(re.findall(r'\bthe\b', search_words, re.IGNORECASE))
        # count_and = len(re.findall(r'\band\b', search_words, re.IGNORECASE))
        return count_the >= 1

    def add_string(example):
        words = example['text'].split()
        new_words = []
        for word in words:
            new_words.append(word)
            if word.lower() == 'the':
                for _ in range(3):
                    new_words.append(string)
        example['text'] = ' '.join(new_words)
        return example

    for label in range(num_cls):
        # Filter data for each class and contains certain patterns
        filtered_train_valid = dataset['train'].filter(lambda x: x['label'] == label and filter_sentence(x))
        #  Select n_samples from filtered data
        train_valid_data = filtered_train_valid.shuffle(seed=sampler_seed).select(range(n_samples))
        # Split the selected data into (ratio) train and (1-ratio) validation
        train_size = int(ratio * n_samples)
        train_data = train_valid_data.select(range(train_size))
        val_data = train_valid_data.select(range(train_size, n_samples))

        # Select test data with a fixed random seed
        filtered_test = dataset['test'].filter(lambda x: x['label'] == label and filter_sentence(x))
        test_data = filtered_test.shuffle(seed=42).select(range(test_per_class))

        train_data = train_data.map(add_string)
        val_data = val_data.map(add_string)
        test_data = test_data.map(add_string)
        
        train_datasets.append(train_data)
        val_datasets.append(val_data)
        test_datasets.append(test_data)

    train_dataset = concatenate_datasets(train_datasets).shuffle(seed=sampler_seed)
    val_dataset = concatenate_datasets(val_datasets).shuffle(seed=sampler_seed)
    test_dataset = concatenate_datasets(test_datasets).shuffle(seed=sampler_seed)
    
    new_dataset = DatasetDict({
        'train':train_dataset,
        'validation':val_dataset,
        'test':test_dataset,
    })

    return new_dataset

N = 5000
data_seed = 5
max_words= 40

ds1 = load_dataset("fancyzhx/yelp_polarity")
ds2 = load_dataset("fancyzhx/amazon_polarity")
ds2 = ds2.rename_column("content", "text")
ds2 = ds2.remove_columns("title")

set_seed(data_seed)
ds1_sample = realworld_data_sampler(ds1, N, data_seed, max_search_words=max_words, string='yelp.xxx')
set_seed(data_seed)
ds2_sample = realworld_data_sampler(ds2, N, data_seed, max_search_words=max_words, string='amazon.yyy')

new_classlabel = ClassLabel(names=['1', '2'])

# Re-define the dataset with the updated label feature for ds1
ds2_sample = ds2_sample.cast_column('label', new_classlabel)

num_cls = len(set(ds1['test']['label']))

ds1_sample

In [ ]:
ds2_sample['train'][:10]

In [ ]:
ds1_sample['train'][:10]

In [ ]:
def create_realworld_dataset(dataset1, dataset2, random_seed, train_prob=0.90, test_probs=[0.90, 0.7, 0.5, 0.3, 0.1]):
    num_cls = len(set(dataset1['test']['label']))
    def merge_data(split1, split2, ratio):
        merged_data = []
        for label in range(num_cls):
            ds1_label = copy.copy(split1).filter(lambda x: x['label'] == label)
            ds2_label = copy.copy(split2).filter(lambda x: x['label'] == label)

            # print(f"Label {label}:")
            # print(f"Dataset 1 available samples: {len(ds1_label)}")
            # print(f"Dataset 2 available samples: {len(ds2_label)}")
        
            
            total_samples = len(ds1_label)
            if label == 0:
                ds1_samples = int(total_samples * ratio)
                ds2_samples = total_samples - ds1_samples
            else:
                ds2_samples = int(total_samples * ratio)
                ds1_samples = total_samples - ds2_samples

            # print(f"Dataset 1 selected samples: {ds1_samples}")
            # print(f"Dataset 2 selected samples: {ds2_samples}")
            
            merged_data.append(ds1_label.select(range(ds1_samples)))
            merged_data.append(ds2_label.select(range(ds2_samples)))
            
            # print(f"Total samples for this label: {min(ds1_samples, len(ds1_label)) + min(ds2_samples, len(ds2_label))}")
            # print("---")
        
        
        result = concatenate_datasets(merged_data)
        # print(f"Total samples in merged dataset: {len(result)}")
        return result

    # Process train and validation sets
    train_dataset = merge_data(dataset1['train'], dataset2['train'], train_prob).shuffle(seed=random_seed)
    val_dataset = merge_data(dataset1['validation'], dataset2['validation'], train_prob).shuffle(seed=random_seed)

    # Process test sets
    test_datasets = []
    for prob in test_probs:
        test_datasets.append(merge_data(dataset1['test'], dataset2['test'], prob).shuffle(seed=random_seed))

    # Return a new DatasetDict with all processed datasets
    return DatasetDict({
        'train': train_dataset,
        'validation': val_dataset,
        'test_in_distribution': test_datasets[0],
        'test_change_70': test_datasets[1],
        'test_balanced_50': test_datasets[2],
        'test_change_30': test_datasets[3],
        'test_flip': test_datasets[4]
    })

In [ ]:


ds = create_realworld_dataset(ds1_sample, ds2_sample, data_seed)
ds


In [ ]:
ds['train'][:10]

## Source-correlated sentiment construction

The sampler inserts three copies of `yelp.xxx` or `amazon.yyy` after each whitespace-delimited token equal to `the` (case-insensitively), according to the review's source. Its filtering step searches for a whole-word `the` within the first `max_search_words` **characters** of whitespace-normalized text. This describes the existing implementation; despite the parameter name, the filtering limit counts characters.

For each class, the merged split retains as many examples as that class has in one source split. In training and validation, negative reviews (label `0`) are 90% Yelp and 10% Amazon, while positive reviews (label `1`) are 10% Yelp and 90% Amazon. Test conditions set the corresponding source-label association to 90%, 70%, 50%, 30%, and 10%, using the same underlying held-out source pools. This notebook has five test conditions and no unmodified-text test split.


In [ ]:
ds['validation'][:10]

In [ ]:
ds['test_in_distribution'][:10]

In [ ]:
print(ds['train']['label'][:10])
print(ds['train']['label'][-10:])

print(ds['validation']['label'][:10])
print(ds['validation']['label'][-10:])

print(ds['test_in_distribution']['label'][:10])
print(ds['test_in_distribution']['label'][-10:])


In [ ]:
class PairedDataset(Dataset):
    def __init__(self, dataset, train_dataset):
        assert len(train_dataset) >= len(dataset)
        self.dataset = dataset
        self.indices = list(range(len(dataset)))
        random.shuffle(self.indices)

        # balance the train_dataset
        train_label_indices = {0: [], 1: []}
        for idx, item in enumerate(train_dataset):
            label = item['label']
            train_label_indices[label].append(idx)
        num_each_cls = len(dataset) // 2
        balanced_indices = []
        for label in [0, 1]:
            balanced_indices.extend(random.sample(train_label_indices[label], k=num_each_cls))
        random.shuffle(balanced_indices)
        self.train_dataset = train_dataset.select(balanced_indices)

        # Pre-compute indices for each label
        self.label_indices = {}
        for idx, item in enumerate(dataset):
            label = item['label']
            if label not in self.label_indices:
                self.label_indices[label] = []
            self.label_indices[label].append(idx)
        

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):

        item1 = self.dataset[idx]
        label = item1['label']
        item4 = self.train_dataset[idx]
        # Find another two items with the same label
        same_label_indices = self.label_indices[label].copy()
        same_label_indices.remove(idx)
        
        if len(same_label_indices) >= 2:
            idx2 = random.choice(same_label_indices)
            item2 = self.dataset[idx2]
            same_label_indices.remove(idx2)
            idx3 = random.choice(same_label_indices)
            item3 = self.dataset[idx3]
        else:
            # Fallback to the same item if no other with same label
            item2 = item1  
            item3 = item1
        return {
            'input_ids1': item1['input_ids'],
            'attention_mask1': item1['attention_mask'],
            'input_ids2': item2['input_ids'],
            'attention_mask2': item2['attention_mask'],
            'input_ids3': item3['input_ids'],
            'attention_mask3': item3['attention_mask'],
            'input_ids4': item4['input_ids'],
            'attention_mask4': item4['attention_mask'],
            'label': label
        }


class PairedDataCollator(DataCollatorWithPadding):
    def __call__(self, features):
        batch1 = super().__call__([{'input_ids': f['input_ids1'], 'attention_mask': f['attention_mask1']} for f in features])
        batch2 = super().__call__([{'input_ids': f['input_ids2'], 'attention_mask': f['attention_mask2']} for f in features])
        batch3 = super().__call__([{'input_ids': f['input_ids3'], 'attention_mask': f['attention_mask3']} for f in features])
        batch4 = super().__call__([{'input_ids': f['input_ids4'], 'attention_mask': f['attention_mask4']} for f in features])
        labels = torch.tensor([f['label'] for f in features])
        
        return {
            'input_ids1': batch1['input_ids'],
            'attention_mask1': batch1['attention_mask'],
            'input_ids2': batch2['input_ids'],
            'attention_mask2': batch2['attention_mask'],
            'input_ids3': batch3['input_ids'],
            'attention_mask3': batch3['attention_mask'],
            'input_ids4': batch4['input_ids'],
            'attention_mask4': batch4['attention_mask'],
            'labels': labels
        }

In [ ]:
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModel.from_pretrained(model_name)

# Tokenize the datasets
def tokenize_function(example):
    return tokenizer(example['text'], max_length=max_words+10, truncation=True)

train_dataset = ds['train'].map(tokenize_function, batched=True)
val_dataset = ds['validation'].map(tokenize_function, batched=True)
test_dataset_ID = ds['test_in_distribution'].map(tokenize_function, batched=True)
test_dataset_OOD_change_70 = ds['test_change_70'].map(tokenize_function, batched=True)
test_dataset_OOD_balanced_50 = ds['test_balanced_50'].map(tokenize_function, batched=True)
test_dataset_OOD_change_30 = ds['test_change_30'].map(tokenize_function, batched=True)
test_dataset_OOD_flip = ds['test_flip'].map(tokenize_function, batched=True)
# test_dataset_OOD_original = synthetic_yelp_ds['test_original'].map(tokenize_function, batched=True)

In [ ]:
paired_train_dataset = PairedDataset(train_dataset, train_dataset)
paired_validation_dataset = PairedDataset(val_dataset, train_dataset)
paired_test_dataset_ID = PairedDataset(test_dataset_ID, train_dataset) 
paired_test_dataset_OOD_change_70 = PairedDataset(test_dataset_OOD_change_70, train_dataset)
paired_test_dataset_OOD_balanced_50 = PairedDataset(test_dataset_OOD_balanced_50, train_dataset)
paired_test_dataset_OOD_change_30 = PairedDataset(test_dataset_OOD_change_30, train_dataset)
paired_test_dataset_OOD_flip = PairedDataset(test_dataset_OOD_flip, train_dataset)
# paired_test_dataset_OOD_original = PairedDataset(test_dataset_OOD_original)

paired_data_collator = PairedDataCollator(tokenizer=tokenizer)

In [ ]:
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    percentage_trainable = 100 * trainable_params / total_params
    
    print(f"Total Parameters: {total_params}")
    print(f"Trainable Parameters: {trainable_params}")
    print(f"Percentage of Trainable Parameters: {percentage_trainable:.4f}%")

def plot_training_metrics(trainer):
    metrics = trainer.state.log_history
    steps = [m['step'] for m in metrics if 'loss' in m]
    train_loss = [m['loss'] for m in metrics if 'loss' in m]
    eval_loss = [m['eval_loss'] for m in metrics if 'eval_loss' in m]
    plt.figure(figsize=(10, 6))
    plt.plot(steps, train_loss, label='Training Loss')
    plt.plot(steps, eval_loss[:len(steps)], label='Validation Loss')
    plt.xlabel('Steps')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

def compute_metrics(eval_preds):
    metric = evaluate.load("f1")  # Load the F1 metric
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels, average='macro')

# SFT0

In [ ]:
class Sft0_Model(nn.Module):
    def __init__(self, base_model, num_labels):
        super().__init__()

        self.base_model = copy.deepcopy(base_model)
        self.classifier = nn.Linear(self.base_model.config.hidden_size, num_labels)

        self.cross_entropy_loss = nn.CrossEntropyLoss()

        for param in self.base_model.parameters():
            param.requires_grad = False

        print(f'Initialise SFT-0 model "{model_name}" (unfreezed all layers) with a linear head!')
        count_parameters(self)

    def forward(self, input_ids1, attention_mask1, input_ids2, attention_mask2, input_ids3, attention_mask3, input_ids4, attention_mask4, labels=None):
        outputs = self.base_model(input_ids1, attention_mask=attention_mask1)
        pooled_output = outputs[1]
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss = self.cross_entropy_loss(logits, labels)
        
        return (loss, logits) if loss is not None else logits
    
set_seed(data_seed)
model = Sft0_Model(base_model, num_cls)

# Define training arguments and trainer
training_args = TrainingArguments(
    overwrite_output_dir=True,
    output_dir=f'./results/sentiment/sft-0/{N}-shot-{data_seed}',
    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",
    logging_steps=0.1,
    save_steps=0.1,
    learning_rate=5e-4,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=10,
    seed=data_seed,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",  # Choose model based on validation loss
    greater_is_better=False,  # Lower validation loss is better
    save_total_limit=1,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=paired_train_dataset,
    eval_dataset=paired_validation_dataset,
    tokenizer=tokenizer,
    data_collator=paired_data_collator,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()


results = {}
train_results = trainer.evaluate(paired_train_dataset)
print(f'train: {train_results}')
results['train'] = train_results

valid_results = trainer.evaluate(paired_validation_dataset)
print(f'validation: {valid_results}')
results['valid'] = valid_results

# Evaluate on the test set
test_results_ID = trainer.evaluate(paired_test_dataset_ID)
print(f'ID: {test_results_ID}')
results['ID'] = test_results_ID

test_results_OOD_change_70 = trainer.evaluate(paired_test_dataset_OOD_change_70)
print(f'OOD change 70: {test_results_OOD_change_70}')
results['OOD change 70'] = test_results_OOD_change_70

test_results_OOD_balanced = trainer.evaluate(paired_test_dataset_OOD_balanced_50)
print(f'OOD balanced: {test_results_OOD_balanced}')
results['OOD balanced'] = test_results_OOD_balanced

test_results_OOD_change_30 = trainer.evaluate(paired_test_dataset_OOD_change_30)
print(f'OOD change 30: {test_results_OOD_change_30}')
results['OOD change 30'] = test_results_OOD_change_30

test_results_OOD_flip = trainer.evaluate(paired_test_dataset_OOD_flip)
print(f'OOD flip: {test_results_OOD_flip}')
results['OOD flip'] = test_results_OOD_flip

# test_results_OOD_original = trainer.evaluate(paired_test_dataset_OOD_original)
# print(f'OOD original: {test_results_OOD_original}')
# results['OOD original'] = test_results_OOD_original


# Manually save the results
with open(f'./results/sentiment/sft-0/{N}-shot-{data_seed}/test_results.json', 'w') as f:
    json.dump(results, f, indent=4)

plot_training_metrics(trainer)

# SFT

In [ ]:
class Sft_Model(nn.Module):
    def __init__(self, base_model, num_labels):
        super().__init__()

        self.base_model = copy.deepcopy(base_model)
        self.classifier = nn.Linear(self.base_model.config.hidden_size, num_labels)
        self.cross_entropy_loss = nn.CrossEntropyLoss()

        print(f'Initialise SFT model "{model_name}" (unfreezed all layers) with a linear head!')
        count_parameters(self)

    def forward(self, input_ids1, attention_mask1, input_ids2, attention_mask2, input_ids3, attention_mask3, input_ids4, attention_mask4, labels=None):
        outputs = self.base_model(input_ids1, attention_mask=attention_mask1)
        pooled_output = outputs[1]
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss = self.cross_entropy_loss(logits, labels)
        
        return (loss, logits) if loss is not None else logits
    
set_seed(data_seed)
model = Sft_Model(base_model, num_cls)

# Define training arguments and trainer
training_args = TrainingArguments(
    overwrite_output_dir=True,
    output_dir=f'./results/sentiment/sft/{N}-shot-{data_seed}',
    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",
    logging_steps=0.1,
    save_steps=0.1,
    learning_rate=5e-5,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=10,
    seed=data_seed,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",  # Choose model based on validation loss
    greater_is_better=False,  # Lower validation loss is better
    save_total_limit=1,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=paired_train_dataset,
    eval_dataset=paired_validation_dataset,
    tokenizer=tokenizer,
    data_collator=paired_data_collator,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()


results = {}
train_results = trainer.evaluate(paired_train_dataset)
print(f'train: {train_results}')
results['train'] = train_results

valid_results = trainer.evaluate(paired_validation_dataset)
print(f'validation: {valid_results}')
results['valid'] = valid_results

# Evaluate on the test set
test_results_ID = trainer.evaluate(paired_test_dataset_ID)
print(f'ID: {test_results_ID}')
results['ID'] = test_results_ID

test_results_OOD_change_70 = trainer.evaluate(paired_test_dataset_OOD_change_70)
print(f'OOD change 70: {test_results_OOD_change_70}')
results['OOD change 70'] = test_results_OOD_change_70

test_results_OOD_balanced = trainer.evaluate(paired_test_dataset_OOD_balanced_50)
print(f'OOD balanced: {test_results_OOD_balanced}')
results['OOD balanced'] = test_results_OOD_balanced

test_results_OOD_change_30 = trainer.evaluate(paired_test_dataset_OOD_change_30)
print(f'OOD change 30: {test_results_OOD_change_30}')
results['OOD change 30'] = test_results_OOD_change_30

test_results_OOD_flip = trainer.evaluate(paired_test_dataset_OOD_flip)
print(f'OOD flip: {test_results_OOD_flip}')
results['OOD flip'] = test_results_OOD_flip

# test_results_OOD_original = trainer.evaluate(paired_test_dataset_OOD_original)
# print(f'OOD original: {test_results_OOD_original}')
# results['OOD original'] = test_results_OOD_original


# Manually save the results
with open(f'./results/sentiment/sft/{N}-shot-{data_seed}/test_results.json', 'w') as f:
    json.dump(results, f, indent=4)

plot_training_metrics(trainer)

# 3: Causal SFT model

In [ ]:
class CausalSft_Model(nn.Module):
    def __init__(self, base_model, num_labels, max_words):
        super().__init__()

        self.max_words = max_words
        self.base_model = copy.deepcopy(base_model)
        self.sft_model = copy.deepcopy(base_model)
        self.sft_classifier = nn.Linear(self.base_model.config.hidden_size, num_labels)
        self.r_2_c = nn.Sequential(
            nn.Linear(self.base_model.config.hidden_size, self.base_model.config.hidden_size // 2),
            nn.ReLU(),
            nn.Linear(self.base_model.config.hidden_size // 2, self.base_model.config.hidden_size // 2),
        )
        self.c_2_c = nn.Sequential(
            nn.Linear(self.base_model.config.hidden_size // 4, self.base_model.config.hidden_size // 4),
            nn.ReLU(),
            nn.Linear(self.base_model.config.hidden_size // 4, self.base_model.config.hidden_size // 4),
        )
        self.num_patches = 10
        self.patches_size = self.max_words // self.num_patches
        self.patch_extractor = nn.Sequential(
            nn.Linear(self.base_model.config.hidden_size * self.num_patches, self.base_model.config.hidden_size * self.num_patches // 2),
            nn.ReLU(),
            nn.Linear(self.base_model.config.hidden_size * self.num_patches // 2, self.base_model.config.hidden_size),
            nn.ReLU(),
            nn.Linear(self.base_model.config.hidden_size, self.base_model.config.hidden_size),
        )
        self.classifier = nn.Linear(self.base_model.config.hidden_size + self.base_model.config.hidden_size // 4, num_labels)

        self.cross_entropy_loss = nn.CrossEntropyLoss()
        self.mse_loss = nn.MSELoss()

        for param in self.base_model.parameters():
            param.requires_grad = False

        print(f'Initialise Causal SFT model "{model_name}" (unfreezed all layers) with a linear head!')
        count_parameters(self)

    def entropy_maximization(self, feature):
        p = F.softmax(feature, dim=-1)
        log_p = F.log_softmax(feature, dim=-1)
        entropy = -torch.mean(torch.mean(p * log_p, dim=-1))  # Maximize entropy
        return -entropy  # Minimize negative entropy

    def forward(self, input_ids1, attention_mask1, input_ids2, attention_mask2, input_ids3, attention_mask3, input_ids4, attention_mask4, labels=None):
        B, _ = input_ids1.size()
        # avoid unwanted computational graph
        outputs_sft2 = self.sft_model(input_ids2, attention_mask=attention_mask2)[1]
        outputs_sft2_logits = self.sft_classifier(outputs_sft2)
        # pull out R0 and R1 using another data point
        outputs_base3 = self.base_model(input_ids3, attention_mask=attention_mask3)[1]
        outputs_sft3 = self.sft_model(input_ids3, attention_mask=attention_mask3)[1]
        # optimise C from R0 and R1
        base_C3= self.r_2_c(outputs_base3)[:, :self.base_model.config.hidden_size//4]
        sft_C3 = self.r_2_c(outputs_sft3)[:, :self.base_model.config.hidden_size//4]
        # gather C 
        outputs_sft = self.sft_model(input_ids1, attention_mask=attention_mask1)[1]
        sft_C = self.r_2_c(outputs_sft)[:, :self.base_model.config.hidden_size//4]
        # extract non-contextual embeddings
        with torch.no_grad():
            embeddings_1 = self.sft_model.embeddings(input_ids1, attention_mask1)[:, :self.max_words]
            embeddings_2 = self.sft_model.embeddings(input_ids2, attention_mask2)[:, :self.max_words]
            embeddings_4 = self.sft_model.embeddings(input_ids4, attention_mask4)[:, :self.max_words]
            
        if self.training:
            input = embeddings_2
            patches = torch.sum(input.view(B, self.num_patches, self.patches_size, -1), dim=2)
            Ai_samples = self.patch_extractor(patches.view(B, -1))
            Ai_samples = Ai_samples[torch.randperm(B).to(input_ids1.device), :]
            Ai_Z1 = torch.cat([Ai_samples, self.c_2_c(sft_C)], dim=-1)
            logits = self.classifier(Ai_Z1)
        else:
            num_samples = 20
            logits = 0
            input = embeddings_1
            patches = torch.sum(input.view(B, self.num_patches, self.patches_size, -1), dim=2)
            Ai_samples = self.patch_extractor(patches.view(B, -1))
            for _ in range(num_samples):
                Ai_samples = Ai_samples[torch.randperm(B).to(input_ids1.device), :]
                Ai_Z1 = torch.cat([Ai_samples, self.c_2_c(sft_C)], dim=-1)
                logits += F.softmax(self.classifier(Ai_Z1), dim=-1)
            logits = logits / num_samples

        loss = None
        if labels is not None:
            loss_sft = self.cross_entropy_loss(outputs_sft2_logits, labels) 
            loss_identify = self.mse_loss(base_C3, sft_C3) + self.entropy_maximization(sft_C3) + self.entropy_maximization(base_C3)
            loss_cls = self.cross_entropy_loss(logits, labels) 
            
            loss = loss_sft + loss_identify + loss_cls 
        
        return (loss, logits) if loss is not None else logits
    
    def forward2(self, input_ids, attention_mask):
        B, _ = input_ids.size()
        R0 = self.base_model(input_ids, attention_mask=attention_mask)[1]
        R1 = self.sft_model(input_ids, attention_mask=attention_mask)[1]
        C = self.r_2_c(R1)[:, :self.base_model.config.hidden_size//4]
        input = self.sft_model.embeddings(input_ids, attention_mask=attention_mask)[:, :self.max_words]
        patches = torch.sum(input.view(B, self.num_patches, self.patches_size, -1), dim=2)
        Ai_samples = self.patch_extractor(patches.view(B, -1))
        num_samples = 20
        Ai_Z1_ = 0
        for _ in range(num_samples):
            Ai_samples = Ai_samples[torch.randperm(B).to(input_ids.device), :]
            Ai_Z1 = torch.cat([Ai_samples, self.c_2_c(C)], dim=-1)
            Ai_Z1_ += Ai_Z1
        Ai_Z1_ = Ai_Z1_ / num_samples
        return (R0, R1, C, Ai_samples, Ai_Z1_)
    
set_seed(data_seed)
model = CausalSft_Model(base_model, num_cls, max_words)

# Define training arguments and trainer
training_args = TrainingArguments(
    overwrite_output_dir=True,
    output_dir=f'./results/sentiment/causal-sft/{N}-shot-{data_seed}',
    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",
    logging_steps=0.1,
    save_steps=0.1,
    learning_rate=5e-5,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=10,
    seed=data_seed,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",  # Choose model based on validation loss
    greater_is_better=False,  # Lower validation loss is better
    save_total_limit=1,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=paired_train_dataset,
    eval_dataset=paired_validation_dataset,
    tokenizer=tokenizer,
    data_collator=paired_data_collator,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()


results = {}
train_results = trainer.evaluate(paired_train_dataset)
print(f'train: {train_results}')
results['train'] = train_results

valid_results = trainer.evaluate(paired_validation_dataset)
print(f'validation: {valid_results}')
results['valid'] = valid_results

# Evaluate on the test set
test_results_ID = trainer.evaluate(paired_test_dataset_ID)
print(f'ID: {test_results_ID}')
results['ID'] = test_results_ID

test_results_OOD_change_70 = trainer.evaluate(paired_test_dataset_OOD_change_70)
print(f'OOD change 70: {test_results_OOD_change_70}')
results['OOD change 70'] = test_results_OOD_change_70

test_results_OOD_balanced = trainer.evaluate(paired_test_dataset_OOD_balanced_50)
print(f'OOD balanced: {test_results_OOD_balanced}')
results['OOD balanced'] = test_results_OOD_balanced

test_results_OOD_change_30 = trainer.evaluate(paired_test_dataset_OOD_change_30)
print(f'OOD change 30: {test_results_OOD_change_30}')
results['OOD change 30'] = test_results_OOD_change_30

test_results_OOD_flip = trainer.evaluate(paired_test_dataset_OOD_flip)
print(f'OOD flip: {test_results_OOD_flip}')
results['OOD flip'] = test_results_OOD_flip


# Manually save the results
with open(f'./results/sentiment/causal-sft/{N}-shot-{data_seed}/test_results.json', 'w') as f:
    json.dump(results, f, indent=4)

plot_training_metrics(trainer)

# 4: Causal SFT no front door

In [ ]:
class CausalSft_Model_No(nn.Module):
    def __init__(self, base_model, num_labels, max_words):
        super().__init__()

        self.max_words = max_words
        self.base_model = copy.deepcopy(base_model)
        self.sft_model = copy.deepcopy(base_model)
        self.sft_classifier = nn.Linear(self.base_model.config.hidden_size, num_labels)
        self.r_2_c = nn.Sequential(
            nn.Linear(self.base_model.config.hidden_size, self.base_model.config.hidden_size // 2),
            nn.ReLU(),
            nn.Linear(self.base_model.config.hidden_size // 2, self.base_model.config.hidden_size // 2),
        )
        self.c_2_c = nn.Sequential(
            nn.Linear(self.base_model.config.hidden_size // 4, self.base_model.config.hidden_size // 4),
            nn.ReLU(),
            nn.Linear(self.base_model.config.hidden_size // 4, self.base_model.config.hidden_size // 4),
        )
        self.num_patches = 10
        self.patches_size = self.max_words // self.num_patches
        self.patch_extractor = nn.Sequential(
            nn.Linear(self.base_model.config.hidden_size * self.num_patches, self.base_model.config.hidden_size * self.num_patches // 2),
            nn.ReLU(),
            nn.Linear(self.base_model.config.hidden_size * self.num_patches // 2, self.base_model.config.hidden_size),
            nn.ReLU(),
            nn.Linear(self.base_model.config.hidden_size, self.base_model.config.hidden_size),
        )
        self.classifier = nn.Linear(self.base_model.config.hidden_size + self.base_model.config.hidden_size // 4, num_labels)

        self.cross_entropy_loss = nn.CrossEntropyLoss()
        self.mse_loss = nn.MSELoss()

        for param in self.base_model.parameters():
            param.requires_grad = False

        print(f'Initialise Causal SFT model "{model_name}" (unfreezed all layers) with a linear head!')
        count_parameters(self)

    def entropy_maximization(self, feature):
        p = F.softmax(feature, dim=-1)
        log_p = F.log_softmax(feature, dim=-1)
        entropy = -torch.mean(torch.mean(p * log_p, dim=-1))  # Maximize entropy
        return -entropy  # Minimize negative entropy

    def forward(self, input_ids1, attention_mask1, input_ids2, attention_mask2, input_ids3, attention_mask3, input_ids4, attention_mask4, labels=None):
        B, _ = input_ids1.size()
        # avoid unwanted computational graph
        outputs_sft2 = self.sft_model(input_ids2, attention_mask=attention_mask2)[1]
        outputs_sft2_logits = self.sft_classifier(outputs_sft2)
        # pull out R0 and R1 using another data point
        outputs_base3 = self.base_model(input_ids3, attention_mask=attention_mask3)[1]
        outputs_sft3 = self.sft_model(input_ids3, attention_mask=attention_mask3)[1]
        # optimise C from R0 and R1
        base_C3= self.r_2_c(outputs_base3)[:, :self.base_model.config.hidden_size//4]
        sft_C3 = self.r_2_c(outputs_sft3)[:, :self.base_model.config.hidden_size//4]
        # gather C 
        outputs_sft = self.sft_model(input_ids1, attention_mask=attention_mask1)[1]
        sft_C = self.r_2_c(outputs_sft)[:, :self.base_model.config.hidden_size//4]
        # extract non-contextual embeddings
        with torch.no_grad():
            embeddings_1 = self.sft_model.embeddings(input_ids1, attention_mask1)[:, :self.max_words]
            embeddings_2 = self.sft_model.embeddings(input_ids2, attention_mask2)[:, :self.max_words]
            embeddings_4 = self.sft_model.embeddings(input_ids4, attention_mask4)[:, :self.max_words]

        input = embeddings_1
        patches = torch.sum(input.view(B, self.num_patches, self.patches_size, -1), dim=2)
        Ai_samples = self.patch_extractor(patches.view(B, -1))
        # Ai_samples = Ai_samples[torch.randperm(B).to(input_ids1.device), :]
        Ai_Z1 = torch.cat([Ai_samples, self.c_2_c(sft_C)], dim=-1)
        logits = self.classifier(Ai_Z1)  

        loss = None
        if labels is not None:
            loss_sft = self.cross_entropy_loss(outputs_sft2_logits, labels) 
            loss_identify = self.mse_loss(base_C3, sft_C3) + self.entropy_maximization(sft_C3) + self.entropy_maximization(base_C3)
            loss_cls = self.cross_entropy_loss(logits, labels) 
            
            loss = loss_sft + loss_identify + loss_cls 
        
        return (loss, logits) if loss is not None else logits
    
set_seed(data_seed)
model = CausalSft_Model_No(base_model, num_cls, max_words)

# Define training arguments and trainer
training_args = TrainingArguments(
    overwrite_output_dir=True,
    output_dir=f'./results/sentiment/causal-sft-no/{N}-shot-{data_seed}',
    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",
    logging_steps=0.1,
    save_steps=0.1,
    learning_rate=5e-5,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=10,
    seed=data_seed,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",  # Choose model based on validation loss
    greater_is_better=False,  # Lower validation loss is better
    save_total_limit=1,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=paired_train_dataset,
    eval_dataset=paired_validation_dataset,
    tokenizer=tokenizer,
    data_collator=paired_data_collator,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()


results = {}
train_results = trainer.evaluate(paired_train_dataset)
print(f'train: {train_results}')
results['train'] = train_results

valid_results = trainer.evaluate(paired_validation_dataset)
print(f'validation: {valid_results}')
results['valid'] = valid_results

# Evaluate on the test set
test_results_ID = trainer.evaluate(paired_test_dataset_ID)
print(f'ID: {test_results_ID}')
results['ID'] = test_results_ID

test_results_OOD_change_70 = trainer.evaluate(paired_test_dataset_OOD_change_70)
print(f'OOD change 70: {test_results_OOD_change_70}')
results['OOD change 70'] = test_results_OOD_change_70

test_results_OOD_balanced = trainer.evaluate(paired_test_dataset_OOD_balanced_50)
print(f'OOD balanced: {test_results_OOD_balanced}')
results['OOD balanced'] = test_results_OOD_balanced

test_results_OOD_change_30 = trainer.evaluate(paired_test_dataset_OOD_change_30)
print(f'OOD change 30: {test_results_OOD_change_30}')
results['OOD change 30'] = test_results_OOD_change_30

test_results_OOD_flip = trainer.evaluate(paired_test_dataset_OOD_flip)
print(f'OOD flip: {test_results_OOD_flip}')
results['OOD flip'] = test_results_OOD_flip


# Manually save the results
with open(f'./results/sentiment/causal-sft-no/{N}-shot-{data_seed}/test_results.json', 'w') as f:
    json.dump(results, f, indent=4)

plot_training_metrics(trainer)

## 5. Causal SFT - C

In [ ]:
class CausalSft_Model_C(nn.Module):
    def __init__(self, base_model, num_labels, max_words):
        super().__init__()

        self.max_words = max_words
        self.base_model = copy.deepcopy(base_model)
        self.sft_model = copy.deepcopy(base_model)
        self.sft_classifier = nn.Linear(self.base_model.config.hidden_size, num_labels)
        self.r_2_c = nn.Sequential(
            nn.Linear(self.base_model.config.hidden_size, self.base_model.config.hidden_size // 2),
            nn.ReLU(),
            nn.Linear(self.base_model.config.hidden_size // 2, self.base_model.config.hidden_size // 2),
        )
        self.c_2_c = nn.Sequential(
            nn.Linear(self.base_model.config.hidden_size // 4, self.base_model.config.hidden_size // 4),
            nn.ReLU(),
            nn.Linear(self.base_model.config.hidden_size // 4, self.base_model.config.hidden_size // 4),
        )
        self.num_patches = 10
        self.patches_size = self.max_words // self.num_patches
        self.patch_extractor = nn.Sequential(
            nn.Linear(self.base_model.config.hidden_size * self.num_patches, self.base_model.config.hidden_size * self.num_patches // 2),
            nn.ReLU(),
            nn.Linear(self.base_model.config.hidden_size * self.num_patches // 2, self.base_model.config.hidden_size),
            nn.ReLU(),
            nn.Linear(self.base_model.config.hidden_size, self.base_model.config.hidden_size),
        )
        # self.c_patch_mixer = nn.Sequential(
        #     nn.Linear(self.base_model.config.hidden_size // 4, self.base_model.config.hidden_size // 8),
        #     nn.ReLU(),
        #     nn.Linear(self.base_model.config.hidden_size // 8, self.base_model.config.hidden_size // 8),
        # )
        self.classifier = nn.Linear(self.base_model.config.hidden_size // 4, num_labels)

        self.cross_entropy_loss = nn.CrossEntropyLoss()
        self.mse_loss = nn.MSELoss()

        for param in self.base_model.parameters():
            param.requires_grad = False

        print(f'Initialise Causal SFT model "{model_name}" (unfreezed all layers) with a linear head!')
        count_parameters(self)

    def entropy_maximization(self, feature):
        p = F.softmax(feature, dim=-1)
        log_p = F.log_softmax(feature, dim=-1)
        entropy = -torch.mean(torch.mean(p * log_p, dim=-1))  # Maximize entropy
        return -entropy  # Minimize negative entropy

    def forward(self, input_ids1, attention_mask1, input_ids2, attention_mask2, input_ids3, attention_mask3, input_ids4, attention_mask4, labels=None):
        B, _ = input_ids1.size()
        # avoid unwanted computational graph
        outputs_sft2 = self.sft_model(input_ids2, attention_mask=attention_mask2)[1]
        outputs_sft2_logits = self.sft_classifier(outputs_sft2)
        # pull out R0 and R1 using another data point
        outputs_base3 = self.base_model(input_ids3, attention_mask=attention_mask3)[1]
        outputs_sft3 = self.sft_model(input_ids3, attention_mask=attention_mask3)[1]
        # optimise C from R0 and R1
        base_C3= self.r_2_c(outputs_base3)[:, :self.base_model.config.hidden_size//4]
        sft_C3 = self.r_2_c(outputs_sft3)[:, :self.base_model.config.hidden_size//4]
        # gather C 
        outputs_sft = self.sft_model(input_ids1, attention_mask=attention_mask1)[1]
        sft_C = self.r_2_c(outputs_sft)[:, :self.base_model.config.hidden_size//4]
        # extract non-contextual embeddings
        with torch.no_grad():
            embeddings_1 = self.sft_model.embeddings(input_ids1, attention_mask1)[:, :self.max_words]
            embeddings_2 = self.sft_model.embeddings(input_ids2, attention_mask2)[:, :self.max_words]
            embeddings_4 = self.sft_model.embeddings(input_ids4, attention_mask4)[:, :self.max_words]

        # input = embeddings_1
        # patches = torch.sum(input.view(B, self.num_patches, self.patches_size, -1), dim=2)
        # Ai_samples = self.patch_extractor(patches.view(B, -1))
        # Ai_samples = Ai_samples[torch.randperm(B).to(input_ids1.device), :]
        Ai_Z1 = self.c_2_c(sft_C)
        logits = self.classifier(Ai_Z1)   

        loss = None
        if labels is not None:
            loss_sft = self.cross_entropy_loss(outputs_sft2_logits, labels) 
            loss_identify = self.mse_loss(base_C3, sft_C3) + self.entropy_maximization(sft_C3) + self.entropy_maximization(base_C3)
            loss_cls = self.cross_entropy_loss(logits, labels) 
            
            loss = loss_sft + loss_identify + loss_cls 
        
        return (loss, logits) if loss is not None else logits
    
set_seed(data_seed)
model = CausalSft_Model_C(base_model, num_cls, max_words)

# Define training arguments and trainer
training_args = TrainingArguments(
    overwrite_output_dir=True,
    output_dir=f'./results/sentiment/causal-sft-c/{N}-shot-{data_seed}',
    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",
    logging_steps=0.1,
    save_steps=0.1,
    learning_rate=5e-5,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=10,
    seed=data_seed,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",  # Choose model based on validation loss
    greater_is_better=False,  # Lower validation loss is better
    save_total_limit=1,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=paired_train_dataset,
    eval_dataset=paired_validation_dataset,
    tokenizer=tokenizer,
    data_collator=paired_data_collator,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()


results = {}
train_results = trainer.evaluate(paired_train_dataset)
print(f'train: {train_results}')
results['train'] = train_results

valid_results = trainer.evaluate(paired_validation_dataset)
print(f'validation: {valid_results}')
results['valid'] = valid_results

# Evaluate on the test set
test_results_ID = trainer.evaluate(paired_test_dataset_ID)
print(f'ID: {test_results_ID}')
results['ID'] = test_results_ID

test_results_OOD_change_70 = trainer.evaluate(paired_test_dataset_OOD_change_70)
print(f'OOD change 70: {test_results_OOD_change_70}')
results['OOD change 70'] = test_results_OOD_change_70

test_results_OOD_balanced = trainer.evaluate(paired_test_dataset_OOD_balanced_50)
print(f'OOD balanced: {test_results_OOD_balanced}')
results['OOD balanced'] = test_results_OOD_balanced

test_results_OOD_change_30 = trainer.evaluate(paired_test_dataset_OOD_change_30)
print(f'OOD change 30: {test_results_OOD_change_30}')
results['OOD change 30'] = test_results_OOD_change_30

test_results_OOD_flip = trainer.evaluate(paired_test_dataset_OOD_flip)
print(f'OOD flip: {test_results_OOD_flip}')
results['OOD flip'] = test_results_OOD_flip


# Manually save the results
with open(f'./results/sentiment/causal-sft-c/{N}-shot-{data_seed}/test_results.json', 'w') as f:
    json.dump(results, f, indent=4)

plot_training_metrics(trainer)

## 6. Causal SFT - Phi

In [ ]:
class CausalSft_Model_Phi(nn.Module):
    def __init__(self, base_model, num_labels, max_words):
        super().__init__()

        self.max_words = max_words
        self.base_model = copy.deepcopy(base_model)
        self.sft_model = copy.deepcopy(base_model)
        self.sft_classifier = nn.Linear(self.base_model.config.hidden_size, num_labels)
        self.r_2_c = nn.Sequential(
            nn.Linear(self.base_model.config.hidden_size, self.base_model.config.hidden_size // 2),
            nn.ReLU(),
            nn.Linear(self.base_model.config.hidden_size // 2, self.base_model.config.hidden_size // 2),
        )
        self.c_2_c = nn.Sequential(
            nn.Linear(self.base_model.config.hidden_size // 4, self.base_model.config.hidden_size // 4),
            nn.ReLU(),
            nn.Linear(self.base_model.config.hidden_size // 4, self.base_model.config.hidden_size // 4),
        )
        self.num_patches = 10
        self.patches_size = self.max_words // self.num_patches
        self.patch_extractor = nn.Sequential(
            nn.Linear(self.base_model.config.hidden_size * self.num_patches, self.base_model.config.hidden_size * self.num_patches // 2),
            nn.ReLU(),
            nn.Linear(self.base_model.config.hidden_size * self.num_patches // 2, self.base_model.config.hidden_size),
            nn.ReLU(),
            nn.Linear(self.base_model.config.hidden_size, self.base_model.config.hidden_size),
        )
        # self.c_patch_mixer = nn.Sequential(
        #     nn.Linear(self.base_model.config.hidden_size, self.base_model.config.hidden_size // 8),
        #     nn.ReLU(),
        #     nn.Linear(self.base_model.config.hidden_size // 8, self.base_model.config.hidden_size // 8),
        # )
        self.classifier = nn.Linear(self.base_model.config.hidden_size, num_labels)

        self.cross_entropy_loss = nn.CrossEntropyLoss()
        self.mse_loss = nn.MSELoss()

        for param in self.base_model.parameters():
            param.requires_grad = False

        print(f'Initialise Causal SFT model "{model_name}" (unfreezed all layers) with a linear head!')
        count_parameters(self)

    def entropy_maximization(self, feature):
        p = F.softmax(feature, dim=-1)
        log_p = F.log_softmax(feature, dim=-1)
        entropy = -torch.mean(torch.mean(p * log_p, dim=-1))  # Maximize entropy
        return -entropy  # Minimize negative entropy

    def forward(self, input_ids1, attention_mask1, input_ids2, attention_mask2, input_ids3, attention_mask3, input_ids4, attention_mask4, labels=None):
        B, _ = input_ids1.size()
        # avoid unwanted computational graph
        outputs_sft2 = self.sft_model(input_ids2, attention_mask=attention_mask2)[1]
        outputs_sft2_logits = self.sft_classifier(outputs_sft2)
        # pull out R0 and R1 using another data point
        outputs_base3 = self.base_model(input_ids3, attention_mask=attention_mask3)[1]
        outputs_sft3 = self.sft_model(input_ids3, attention_mask=attention_mask3)[1]
        # optimise C from R0 and R1
        base_C3= self.r_2_c(outputs_base3)[:, :self.base_model.config.hidden_size//4]
        sft_C3 = self.r_2_c(outputs_sft3)[:, :self.base_model.config.hidden_size//4]
        # gather C 
        outputs_sft = self.sft_model(input_ids1, attention_mask=attention_mask1)[1]
        sft_C = self.r_2_c(outputs_sft)[:, :self.base_model.config.hidden_size//4]
        # extract non-contextual embeddings
        with torch.no_grad():
            embeddings_1 = self.sft_model.embeddings(input_ids1, attention_mask1)[:, :self.max_words]
            embeddings_2 = self.sft_model.embeddings(input_ids2, attention_mask2)[:, :self.max_words]
            embeddings_4 = self.sft_model.embeddings(input_ids4, attention_mask4)[:, :self.max_words]

        input = embeddings_1
        patches = torch.sum(input.view(B, self.num_patches, self.patches_size, -1), dim=2)
        Ai_samples = self.patch_extractor(patches.view(B, -1))
        # Ai_samples = Ai_samples[torch.randperm(B).to(input_ids1.device), :]
        Ai_Z1 = Ai_samples
        logits = self.classifier(Ai_Z1)   

        loss = None
        if labels is not None:
            loss_sft = self.cross_entropy_loss(outputs_sft2_logits, labels) 
            loss_identify = self.mse_loss(base_C3, sft_C3) + self.entropy_maximization(sft_C3) + self.entropy_maximization(base_C3)
            loss_cls = self.cross_entropy_loss(logits, labels) 
            
            loss = loss_sft + loss_identify + loss_cls 
        
        return (loss, logits) if loss is not None else logits
    
set_seed(data_seed)
model = CausalSft_Model_Phi(base_model, num_cls, max_words)

# Define training arguments and trainer
training_args = TrainingArguments(
    overwrite_output_dir=True,
    output_dir=f'./results/sentiment/causal-sft-phi/{N}-shot-{data_seed}',
    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",
    logging_steps=0.1,
    save_steps=0.1,
    learning_rate=5e-5,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=10,
    seed=data_seed,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",  # Choose model based on validation loss
    greater_is_better=False,  # Lower validation loss is better
    save_total_limit=1,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=paired_train_dataset,
    eval_dataset=paired_validation_dataset,
    tokenizer=tokenizer,
    data_collator=paired_data_collator,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()


results = {}
train_results = trainer.evaluate(paired_train_dataset)
print(f'train: {train_results}')
results['train'] = train_results

valid_results = trainer.evaluate(paired_validation_dataset)
print(f'validation: {valid_results}')
results['valid'] = valid_results

# Evaluate on the test set
test_results_ID = trainer.evaluate(paired_test_dataset_ID)
print(f'ID: {test_results_ID}')
results['ID'] = test_results_ID

test_results_OOD_change_70 = trainer.evaluate(paired_test_dataset_OOD_change_70)
print(f'OOD change 70: {test_results_OOD_change_70}')
results['OOD change 70'] = test_results_OOD_change_70

test_results_OOD_balanced = trainer.evaluate(paired_test_dataset_OOD_balanced_50)
print(f'OOD balanced: {test_results_OOD_balanced}')
results['OOD balanced'] = test_results_OOD_balanced

test_results_OOD_change_30 = trainer.evaluate(paired_test_dataset_OOD_change_30)
print(f'OOD change 30: {test_results_OOD_change_30}')
results['OOD change 30'] = test_results_OOD_change_30

test_results_OOD_flip = trainer.evaluate(paired_test_dataset_OOD_flip)
print(f'OOD flip: {test_results_OOD_flip}')
results['OOD flip'] = test_results_OOD_flip


# Manually save the results
with open(f'./results/sentiment/causal-sft-phi/{N}-shot-{data_seed}/test_results.json', 'w') as f:
    json.dump(results, f, indent=4)

plot_training_metrics(trainer)